In [ ]:
# =========================================
# 1. Upload & Import Libraries
# =========================================

import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")

try:
    from google.colab import files
    uploaded = files.upload()
except:
    print("Running locally")

print("Files in directory:", os.listdir())


# =========================================
# 2. Auto Detect Files
# =========================================

def find_file(keys):
    for f in os.listdir():
        fname = f.lower()
        if all(k in fname for k in keys):
            return f
    return None

files_map = {
    'customers': ['customers'],
    'orders': ['orders'],
    'support': ['support'],
    'web': ['web'],
    'churn': ['churn'],
    'campaign': ['intervention']
}

detected = {k: find_file(v) for k, v in files_map.items()}

print("\nDetected files:")
for k, v in detected.items():
    print(f"{k}: {v}")


# =========================================
# 3. Load Data Safely
# =========================================

def safe_load(file):
    if file is None:
        print(f"WARNING: Missing file -> {file}")
        return pd.DataFrame()
    return pd.read_csv(file)

customers = safe_load(detected['customers'])
orders    = safe_load(detected['orders'])
support   = safe_load(detected['support'])
web       = safe_load(detected['web'])
churn     = safe_load(detected['churn'])
campaign  = safe_load(detected['campaign'])


# =========================================
# 4. Feature Engineering
# =========================================

# --- Date conversions ---
if 'order_date' in orders.columns:
    orders['order_date'] = pd.to_datetime(orders['order_date'], errors='coerce')

if 'snapshot_date' in churn.columns:
    churn['snapshot_date'] = pd.to_datetime(churn['snapshot_date'], errors='coerce')

# --- Orders Aggregation ---
order_agg = pd.DataFrame()

if not orders.empty:
    order_agg = orders.groupby('customer_id').agg(
        total_orders=('order_id', 'count'),
        total_spend=('gross_amount', 'sum')
    ).reset_index()

# --- Recency Feature ---
if not orders.empty and not churn.empty:
    last_order = orders.groupby('customer_id')['order_date'].max().reset_index()
    last_order.columns = ['customer_id', 'last_order_date']

    last_order = last_order.merge(
        churn[['customer_id', 'snapshot_date']],
        on='customer_id',
        how='left'
    )

    last_order['recency_days'] = (
        last_order['snapshot_date'] - last_order['last_order_date']
    ).dt.days

    order_agg = order_agg.merge(
        last_order[['customer_id', 'recency_days']],
        on='customer_id',
        how='left'
    )

# --- Support Aggregation ---
support_agg = pd.DataFrame()

if not support.empty:
    support_agg = support.groupby('customer_id').agg(
        ticket_count=('ticket_id', 'count'),
        avg_sentiment=('sentiment_score', 'mean')
    ).reset_index()


# --- Campaign Feature ---
if not campaign.empty:
    if 'last_campaign_received' in campaign.columns:
        campaign['received_campaign'] = campaign['last_campaign_received'].apply(
            lambda x: 0 if str(x).lower() == 'none' else 1
        )
    else:
        campaign['received_campaign'] = 0

    campaign['received_campaign'] = campaign['received_campaign'].fillna(0)


# =========================================
# 5. Merge All Data
# =========================================

df_final = churn.copy()

def safe_merge(df, other):
    return df.merge(other, on='customer_id', how='left') if not other.empty else df

df_final = safe_merge(df_final, customers)
df_final = safe_merge(df_final, order_agg)
df_final = safe_merge(df_final, support_agg)
df_final = safe_merge(df_final, web)
df_final = safe_merge(df_final, campaign)

print("\nFinal Columns:\n", df_final.columns)


# =========================================
# 6. Visualization
# =========================================

def safe_plot(plot_func):
    try:
        plt.figure(figsize=(6,4))
        plot_func()
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print("Plot skipped:", e)


# Check required columns
required_cols = ['churn_next_60d']
missing = [c for c in required_cols if c not in df_final.columns]

if missing:
    print("\nMissing critical columns:", missing)
else:
    # Churn distribution
    safe_plot(lambda: sns.countplot(x='churn_next_60d', data=df_final))

    # Numerical vs churn
    for col in [
        'total_orders',
        'total_spend',
        'recency_days',
        'ticket_count',
        'avg_sentiment',
        'sessions_30d',
        'last_visit_days_ago'
    ]:
        if col in df_final.columns:
            safe_plot(lambda col=col: sns.boxplot(
                x='churn_next_60d', y=col, data=df_final
            ))

    # Categorical plots
    if 'loyalty_tier' in df_final.columns:
        safe_plot(lambda: sns.countplot(
            x='loyalty_tier',
            hue='churn_next_60d',
            data=df_final
        ))

    if 'received_campaign' in df_final.columns:
        safe_plot(lambda: sns.countplot(
            x='received_campaign',
            hue='churn_next_60d',
            data=df_final
        ))


# =========================================
# 7. Summary Output
# =========================================

print("\nDataset Shape:", df_final.shape)
print("\nMissing Values:\n", df_final.isnull().sum().sort_values(ascending=False).head(10))

print("\nDone ✅")